# CISM Tutorial 03b: Soft Motif Multi-Objective Selection

This notebook starts from the serialized artifact saved at the end of tutorial 02 and ranks motifs with the multi-objective soft motif selection framework.


The flow is:

1. Load the CISM/discriminator artifact from tutorial 02.
2. Define the soft-selection search space.
3. Use Optuna to optimize the additive score weights against leave-one-patient-out Random Forest ROC AUC.
4. Re-score all motifs with the best weights and inspect the selected motif panel.
5. Validate the selected panel with leakage-safe LOOCV.
6. Visualize score components, model feature importance, and the top 3 motifs.

We use the `discriminator` object because it owns both the prepared CISM motif table and the patient class mapping. The raw `CISM` object only knows about motif occurrences; it does not know which patients belong to which disease-state labels.


In [ ]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm

from cism import (
    InferenceFC,
    MotifSelectionWeights,
    SoftMotifSelectionConfig,
    StabilityGateConfig,
    evaluate_soft_motif_selection_loocv,
    optimize_soft_motif_weights,
    score_soft_motifs_from_discriminator,
)
from cism.vis.draw import draw_motif

sns.set_theme(style="whitegrid")


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "tutorials":
    PROJECT_ROOT = PROJECT_ROOT.parent

TUTORIAL_RUNTIME_DIR = PROJECT_ROOT / "tutorials" / "runtime"
SERIALIZED_ROOT = TUTORIAL_RUNTIME_DIR / "serialized"
SOFT_MOTIF_OUTPUT_ROOT = TUTORIAL_RUNTIME_DIR / "soft_motif_selection"
SOFT_MOTIF_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Serialized artifact directory: {SERIALIZED_ROOT}")
print(f"Soft motif output directory: {SOFT_MOTIF_OUTPUT_ROOT}")


## Load The Artifact From Tutorial 02

Choose the `.pkl` created in tutorial 02. The artifact contains both the `CISM` object and the `TissueStateDiscriminativeMotifs` discriminator created from it.


In [ ]:
# Replace with the artifact file created in tutorial 02.
artifact_path = SERIALIZED_ROOT / "TNBC_cism_ready.pkl"

with open(artifact_path, "rb") as handle:
    artifact = pickle.load(handle)

artifact.keys()


In [ ]:
cism = artifact["cism"]
discriminator = artifact["discriminator"]
labels = artifact["labels"]
network_dataset_root_path = artifact.get("network_dataset_root_path")
dataset_folder = artifact.get("dataset_folder")
tissue_state_csv_path = artifact.get("tissue_state_csv_path")
common_cells_type = artifact.get("common_cells_type")

print(f"network_dataset_root_path = {network_dataset_root_path}")
print(f"dataset_folder = {dataset_folder}")
print(f"tissue_state_csv_path = {tissue_state_csv_path}")
print(f"labels = {labels}")
print(f"patients with labels = {discriminator.get_patients_class(labels).shape[0]}")
print(f"motif rows = {cism.motifs_dataset.shape[0]:,}")
print(f"number of cell types = {len(common_cells_type)}")


## Configure The Search

The stability gate is kept fixed, while Optuna searches for the five additive weights. The absolute scale of the weights is not important because the additive score is normalized by the total weight.


In [ ]:
base_selection_config = SoftMotifSelectionConfig(
    labels=labels,
    top_k=10,
    weights=MotifSelectionWeights(),
    gate=StabilityGateConfig(tau=0.6, gamma=2.0),
    fanmod_p_value_threshold=0.05,
)

N_OPTUNA_TRIALS = 30
WEIGHT_RANGE = (0.0, 4.0)
RANDOM_STATE = 0

base_selection_config


## Optimize Additive Weights With Optuna

Each trial evaluates a complete soft-motif selection configuration in leave-one-patient-out Random Forest validation. This can take a while on a large motif table, so the Optuna progress bar is enabled.


In [ ]:
weight_tuning_result = optimize_soft_motif_weights(
    discriminator=discriminator,
    base_config=base_selection_config,
    n_trials=N_OPTUNA_TRIALS,
    metric="roc_auc",
    random_state=RANDOM_STATE,
    weight_range=WEIGHT_RANGE,
    show_progress_bar=True,
)

selection_config = weight_tuning_result.best_config
best_weights = weight_tuning_result.best_weights

print(f"Best Optuna ROC AUC: {weight_tuning_result.best_score:.3f}")
print(best_weights)
display(weight_tuning_result.trials_dataframe().sort_values("value", ascending=False).head(10))


In [ ]:
weight_df = pd.Series(best_weights.as_dict(), name="weight").reset_index().rename(columns={"index": "component"})

plt.figure(figsize=(7, 4))
sns.barplot(data=weight_df, x="component", y="weight", color="#4C78A8")
plt.title("Optuna-selected additive score weights")
plt.xlabel("")
plt.ylabel("Weight")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()


## Score Motifs With The Optimized Configuration

The final score is `stability_gate * additive_score`. The additive score contains effect size, abundance, prevalence, confidence, and dispersion desirability.


In [ ]:
soft_result = score_soft_motifs_from_discriminator(
    discriminator=discriminator,
    config=selection_config,
)

scored_motifs_df = soft_result.scores
selected_motif_ids = soft_result.selected_motif_ids

display(scored_motifs_df.head(20))
selected_motif_ids


In [ ]:
plot_columns = [
    "effect_score",
    "abundance_score",
    "prevalence_score",
    "confidence_score",
    "dispersion_desirability_score",
    "stability_gate",
    "final_score",
]

top_plot_df = scored_motifs_df.head(10).melt(
    id_vars="ID",
    value_vars=plot_columns,
    var_name="component",
    value_name="score",
)

plt.figure(figsize=(13, 5))
sns.barplot(data=top_plot_df, x="ID", y="score", hue="component")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.title("Top soft motif score components")
plt.tight_layout()


## Leakage-Safe LOOCV Validation

This validation re-selects motifs inside each training fold, trains a Random Forest on the selected motif frequencies, and tests only on the held-out patient.


In [ ]:
loocv_result = evaluate_soft_motif_selection_loocv(
    discriminator=discriminator,
    config=selection_config,
    random_state=RANDOM_STATE,
    show_progress=True,
)

display(loocv_result.results)
print(f"ROC AUC: {loocv_result.get_roc_auc_score():.3f}")
loocv_result.get_metrics()


## Feature Importance

The validation result stores SHAP explanations for each held-out patient. We aggregate mean absolute SHAP values across folds to see which selected motifs most influenced Random Forest predictions.


In [ ]:
def fold_shap_importance(row):
    shap_values, instance = row["shape_values"]
    values = np.asarray(shap_values.values)
    if values.ndim == 3:
        values = np.abs(values).mean(axis=(0, 2))
    elif values.ndim == 2:
        values = np.abs(values).mean(axis=0)
    else:
        values = np.abs(values).reshape(-1)
    return pd.Series(values, index=instance.columns, dtype=float)

importance_df = pd.DataFrame(
    [fold_shap_importance(row) for _, row in loocv_result.results.iterrows()]
).fillna(0.0)

feature_importance = importance_df.mean(axis=0).sort_values(ascending=False)
feature_importance_df = feature_importance.rename("mean_abs_shap").reset_index().rename(columns={"index": "ID"})
feature_importance_df["ID"] = feature_importance_df["ID"].astype(str)

display(feature_importance_df.head(20))

plt.figure(figsize=(9, 6))
sns.barplot(data=feature_importance_df.head(20), y="ID", x="mean_abs_shap", color="#59A14F")
plt.title("Random Forest motif feature importance")
plt.xlabel("Mean absolute SHAP value")
plt.ylabel("Motif ID")
plt.tight_layout()


## Visualize The Top 3 Meaningful Motifs

Here, “meaningful” means the highest final-scoring motifs after stability gating. The table shows their score components before drawing their motif graphs.


In [ ]:
top3_motifs = scored_motifs_df.head(3).copy()
motif_lookup = (
    discriminator.cism.motifs_dataset
    .drop_duplicates("ID")
    .set_index("ID")["motif"]
)

display(top3_motifs[[
    "ID",
    "final_score",
    "additive_score",
    "stability_gate",
    "effect_score",
    "overall_prevalence",
    "direction",
    "logfc_group1_vs_group0",
]])

for _, motif_row in tqdm(top3_motifs.iterrows(), total=top3_motifs.shape[0], desc="Drawing top motifs"):
    motif_id = motif_row["ID"]
    print(
        f"Motif {motif_id}: final={motif_row['final_score']:.3f}, "
        f"effect={motif_row['effect_score']:.3f}, direction={motif_row['direction']}"
    )
    draw_motif(motif_lookup.loc[motif_id], common_cells_type)


## Reuse Selected Motifs With Existing CISM Analysis

`InferenceFC` is the bridge back into CISM's existing validation/inference API. It fixes the selected motif IDs instead of discovering a new motif set.


In [ ]:
soft_inference_feature_conf = soft_result.to_inference_feature_config()

# Equivalent explicit construction:
# soft_inference_feature_conf = InferenceFC(labels=labels, motifs_ids=selected_motif_ids)

soft_inference_feature_conf


In [ ]:
scored_motifs_path = SOFT_MOTIF_OUTPUT_ROOT / "soft_motif_scores.csv"
selected_motifs_path = SOFT_MOTIF_OUTPUT_ROOT / "selected_soft_motif_ids.csv"
optuna_trials_path = SOFT_MOTIF_OUTPUT_ROOT / "soft_motif_weight_optuna_trials.csv"

scored_motifs_df.to_csv(scored_motifs_path, index=False)
pd.Series(selected_motif_ids, name="ID").to_csv(selected_motifs_path, index=False)
weight_tuning_result.trials_dataframe().to_csv(optuna_trials_path, index=False)

print(f"Saved scored motifs: {scored_motifs_path}")
print(f"Saved selected motif IDs: {selected_motifs_path}")
print(f"Saved Optuna trials: {optuna_trials_path}")
